# 05 — LSTM Autoencoder

## 1. Objective

Develop an LSTM autoencoder for unsupervised anomaly detection on the SWaT multivariate time series.

The model will be trained using normal calibration data and will learn to reconstruct temporal patterns in the sensor measurements. Reconstruction error will then provide an anomaly score: observations that are poorly reconstructed relative to the learned normal behaviour may indicate anomalous behaviour.

The implementation follows the LSTM autoencoder approach used in the primary reference notebook, while maintaining the project's requirements for chronological data handling, separation of calibration and evaluation data, reproducibility, and prevention of data leakage.

Attack labels will not be used during model training or reconstruction-error threshold calibration.

### Scope of Evaluation

This notebook evaluates the LSTM autoencoder on the normal validation period using the same eligible-normal split as Notebook 04. Attack-side evaluation — including confusion-matrix counts, precision, recall, and F1 on attack-labelled sequences — is performed in a subsequent stage, following the deferral recorded in Notebook 04 §13.3. The temporal overlap between the attack file and the normal time windows, established by Notebook 02's normal–attack timestamp overlap analysis, must be respected when that evaluation is carried out.

## 2. Methodology

An autoencoder is trained to reconstruct input observations from a compressed internal representation. When trained only on normal process behaviour, the model is expected to learn the temporal patterns present in normal operation.

For an input sequence X, the LSTM encoder maps the sequence to a lower-dimensional representation:

X → Encoder → Z

The decoder then reconstructs the sequence:

Z → Decoder → X̂

The reconstruction error measures the difference between the original and reconstructed sequence. For a sequence containing T time steps and D sensor features, the mean squared reconstruction error is:

MSE = (1 / (T × D)) × Σ (Xₜ,ⱼ − X̂ₜ,ⱼ)²

where:

- Xₜ,ⱼ is the observed value for feature j at time step t.
- X̂ₜ,ⱼ is the corresponding reconstructed value.
- T is the sequence length.
- D is the number of sensor features.

A threshold derived from the reconstruction-error distribution on held-out normal calibration observations is then used to classify sequences as normal or anomalous. Following the percentile approach used by the simple statistical baseline in Notebook 04 §12, the threshold is set at the 99th percentile of reconstruction errors on a portion of the calibration period held out from model training. The threshold is fixed before any attack sequence is scored, and attack labels are not used to select or adjust it. The specific hold-out split used for threshold calibration is recorded alongside the model-training code.

### Temporal Windowing

The model operates on sequences rather than isolated observations. Each sequence contains a fixed number of consecutive observations while preserving their chronological order.

The primary reference implementation uses a window size of 30 observations. This project will initially retain that value so that the implementation remains comparable to the reference, unless the data or modelling results provide evidence that a change is necessary.

Training sequences are generated from normal calibration data only. Normal-validation sequences are chronologically disjoint from the training sequences. Attack sequences are evaluated separately and are known to overlap the calibration and validation time windows, even though they share no exact timestamps with them. Attack-side evaluation is therefore treated as a distinct, time-window-overlapping evaluation, not as a temporally independent hold-out.

The training and normal-validation sequences use the eligible-normal split established in Notebook 04 §9: 316,238 calibration observations (2015-12-28 10:00:00 → 2016-01-01 15:51:19) and 79,060 normal-validation observations (2016-01-01 15:51:20 → 2016-01-02 14:59:59), drawn from the 395,298 eligible normal observations. No re-splitting is performed in this notebook, so reconstruction-error statistics remain directly comparable to the baselines evaluated in Notebook 04.

Sequence generation respects the calibration/validation boundary: the last training sequence ends strictly before the first validation observation, so no validation observation contributes to the reconstruction-error statistics on the calibration data. Within the calibration period, sequences are generated with the configured stride; at the boundary, sequences are truncated to avoid crossing into the validation period.

### Input Feature Contract

Notebook 04 §12.5 documented that nine sensors — `P202`, `P204`, `P206`, `P301`, `P401`, `P404`, `P502`, `P601`, `P603` — had zero standard deviation during the calibration period, and Notebook 03 §6.2 identified six of these as globally constant across both the normal and attack datasets. Zero-variance features produce no reconstruction-error signal and contribute no gradient to the autoencoder.

This notebook excludes the nine constant-in-calibration sensors from the LSTM input, giving a 42-feature input contract. This choice mirrors the 42-feature evaluation set used by the simple statistical threshold baseline in Notebook 04 §12.5 and eliminates zero-gradient inputs from the training signal. The 42 retained features are the remaining sensor columns after the nine excluded features are removed. The input contract is recorded here so that the anomaly scores reported subsequently are reproducible.

### Distribution-Shift Risk

Notebook 04 §10.4 found that the rule-based detector, calibrated on the first 80% of the eligible normal data, flagged 92.71% of the remaining 20% of normal observations. This indicates a substantial shift in the marginal distributions of several sensors between the calibration and validation periods. An LSTM autoencoder trained on calibration data may produce systematically higher reconstruction errors on the validation period, in which case a threshold calibrated directly on the training reconstruction errors would over-flag normal validation observations.

To mitigate this risk, the anomaly threshold is calibrated on a portion of the calibration period held out from model training, so that the threshold reflects the model's reconstruction behaviour on data not seen during training. The distribution of reconstruction errors on the normal validation period is reported alongside the threshold so that the extent of any residual shift remains visible.

### Data Leakage Prevention

The preprocessing parameters used by the model must be learned from the normal calibration data only. Validation and attack observations must not influence the fitted scaler or the model parameters during training.

The final anomaly threshold is also established without using attack labels for threshold selection. All threshold values are derived from held-out normal-calibration reconstruction errors.

## 3. Reproducibility and Model Configuration

The LSTM autoencoder will use fixed random seeds and explicit model parameters so that the experiment can be reproduced.

The initial configuration follows the primary reference implementation:

| Parameter | Value |
|:--|--:|
| Random seed | 42 |
| Sequence window | 30 observations |
| Training stride | 5 |
| Evaluation stride | 1 |
| Hidden size | 64 |
| LSTM layers | 2 |
| Bottleneck size | 16 |
| Dropout | 0.2 |
| Loss function | Mean Squared Error |
| Optimizer | Adam |
| Initial learning rate | 0.001 |
| Batch size | 256 |
| Learning-rate reduction factor | 0.5 |
| Learning-rate scheduler patience | 3 epochs |
| Early-stopping patience | 5 epochs |

These values provide the initial model configuration and are not being treated as universally optimal hyperparameters.

Any change to the configuration will be recorded and justified by the observed modelling results rather than by tuning against final attack labels.